# 4. 탑다운 백테스팅 분석

**전략 구조:**  
글로벌 매크로 신호 → WICS 섹터 선택 → FA 7지표 종목 선택

```
[1단계] 매크로 국면 (data/asset)
  구리·BDI → 경기 국면 / CPI·TNX → 금리 환경
  → A(Risk-On+저금리) / B(Risk-On+고금리) / C(Risk-Off+저금리) / D(Risk-Off+고금리)

[2단계] 섹터 선택 (WICS)
  국면별 섹터 비중 결정

[3단계] 종목 선택 (WICS + DART)
  FA 7지표 점수 상위 N개 종목 편입
```

In [ ]:
import sys
sys.path.insert(0, '..')  # analysis_good593/ 를 패스에 추가

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib

from backtesting.runner import run_backtest
from backtesting.strategy.macro_signal import compute_macro_signals
from backtesting.metrics import calc_metrics, plot_equity_curve, plot_monthly_heatmap, plot_drawdown
from backtesting.report import print_summary_table, plot_full_report, plot_rebalance_history

print('모듈 로딩 완료')

---
## 1. 매크로 국면 시각화

In [ ]:
YEARS = [2022, 2023, 2024]

macro = compute_macro_signals(YEARS)
regime_series = macro.to_series()

# 국면 분포
print('=== 국면 분포 ===')
print(regime_series.value_counts().sort_index())
print()
print('국면 의미:')
print('  A: Risk-On  + 저금리 → IT·경기소비재·산업재')
print('  B: Risk-On  + 고금리 → 에너지·소재·금융')
print('  C: Risk-Off + 저금리 → 헬스케어·유틸리티·필수소비재')
print('  D: Risk-Off + 고금리 → 필수소비재·헬스케어')

---
## 2. 백테스팅 실행

### 2-1. 연도 전체

In [ ]:
# 2022~2024년 전체 백테스팅 (분기별 리밸런싱)
result = run_backtest(
    year=YEARS,
    initial_cash=100_000_000,
    commission=0.00015,
    slippage=0.0005,
    top_n=5,
    verbose=True,
)

print(result.summary())

### 2-2. 특정 분기

In [ ]:
# 2023년 Q3 (7~9월) 백테스팅
result_q = run_backtest(
    year=2023,
    quarter=3,
    initial_cash=100_000_000,
    top_n=5,
    verbose=True,
)
print(result_q.summary())

### 2-3. 특정 월

In [ ]:
# 2024년 11월 백테스팅
result_m = run_backtest(
    year=2024,
    month=11,
    initial_cash=100_000_000,
    top_n=5,
    verbose=True,
)
print(result_m.summary())

---
## 3. 성과 분석

In [ ]:
# 전략 vs 벤치마크 요약 테이블
print_summary_table(result)

In [ ]:
# 자산 곡선
fig, axes = plt.subplots(3, 1, figsize=(13, 12))
plot_equity_curve(result.time_returns, result.benchmark_returns, result.initial_cash, ax=axes[0])
plot_drawdown(result.time_returns, ax=axes[1])
plot_monthly_heatmap(result.time_returns, ax=axes[2])
plt.tight_layout()
plt.show()

In [ ]:
# 리밸런싱 이력
fig = plot_rebalance_history(result)
plt.show()

---
## 4. 전체 리포트 (대시보드)

In [ ]:
fig = plot_full_report(result, macro_signal=macro)
plt.show()

---
## 5. 리밸런싱 날짜별 포트폴리오 확인

In [ ]:
for rebal_date, weights in sorted(result.rebalance_schedule.items()):
    print(f'\n=== {rebal_date} ===')
    for cmp_cd, w in sorted(weights.items(), key=lambda x: -x[1]):
        print(f'  {cmp_cd}: {w:.2%}')